#TALLER VALIDACION DE AGRUPACION
<center>
<strong>Taller Validación Agrupación</strong>
<h2>MINERIA DE DATOS</h2>
PROFESORA ELIZABETH LEON GUZMAN <br />
<strong>TEMA</strong> Agrupación
</center>

-------
####Práctica correspondiente a Validación de la calidad del resultado de modelos de agrupación.
*   SSE
*   SSB
*   Silhouette Coeficient - Coeficiente de silueta
*   Davies-Bouldin index
*   Presicion - recall - F
*   Entropia-Pureza-MI

####Ejemplo tomado de https://scikit-learn.org/

####**Trabajar Individualmente**
--------

1. Ejecutar el ejemplo de https://scikit-learn.org/ para calular el Silhouette Coeficient

In [ ]:
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.datasets import load_iris, make_blobs
from sklearn.metrics import (
    confusion_matrix,
    davies_bouldin_score,
    normalized_mutual_info_score,
    precision_recall_fscore_support,
    silhouette_samples,
    silhouette_score,
)

# Datos sintéticos del ejemplo base de scikit-learn
X, y = make_blobs(
    n_samples=500,
    n_features=2,
    centers=4,
    cluster_std=1,
    center_box=(-10.0, 10.0),
    shuffle=True,
    random_state=1,
)


In [ ]:
range_n_clusters = [2, 3, 4, 5, 6]
sample_results = []

for n_clusters in range_n_clusters:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    ax1.set_xlim([-0.1, 1])
    ax1.set_ylim([0, len(X) + (n_clusters + 1) * 10])

    clusterer = KMeans(n_clusters=n_clusters, n_init=10, random_state=10)
    cluster_labels = clusterer.fit_predict(X)

    silhouette_avg = silhouette_score(X, cluster_labels)
    davies_bouldin = davies_bouldin_score(X, cluster_labels)
    sse = clusterer.inertia_

    sample_results.append(
        {
            "k": n_clusters,
            "silhouette": silhouette_avg,
            "davies_bouldin": davies_bouldin,
            "sse": sse,
        }
    )

    print(
        f"k={n_clusters}: silhouette={silhouette_avg:.4f}, "
        f"Davies-Bouldin={davies_bouldin:.4f}, SSE={sse:.2f}"
    )

    sample_silhouette_values = silhouette_samples(X, cluster_labels)

    y_lower = 10
    for i in range(n_clusters):
        ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]
        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i

        color = cm.nipy_spectral(float(i) / n_clusters)
        ax1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster_silhouette_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
        )
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10

    ax1.set_title(f"Silueta para k={n_clusters}")
    ax1.set_xlabel("Coeficiente de silueta")
    ax1.set_ylabel("Cluster")
    ax1.axvline(x=silhouette_avg, color="red", linestyle="--")
    ax1.set_yticks([])
    ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

    colors = cm.nipy_spectral(cluster_labels.astype(float) / n_clusters)
    ax2.scatter(
        X[:, 0],
        X[:, 1],
        marker=".",
        s=30,
        lw=0,
        alpha=0.7,
        c=colors,
        edgecolor="k",
    )

    centers = clusterer.cluster_centers_
    ax2.scatter(
        centers[:, 0],
        centers[:, 1],
        marker="o",
        c="white",
        alpha=1,
        s=200,
        edgecolor="k",
    )

    for i, center in enumerate(centers):
        ax2.scatter(center[0], center[1], marker=f"${i}$", alpha=1, s=50, edgecolor="k")

    ax2.set_title("Visualización de los grupos")
    ax2.set_xlabel("Característica 1")
    ax2.set_ylabel("Característica 2")

    plt.suptitle(
        f"Análisis de silueta y K-means para k={n_clusters}",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

sample_results_df = pd.DataFrame(sample_results)
print("\nResumen de métricas internas para los datos sintéticos:")
print(sample_results_df.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(sample_results_df["k"], sample_results_df["silhouette"], marker="o")
axes[0].set_title("Silhouette promedio")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Silhouette")
axes[0].grid(alpha=0.3)

axes[1].plot(sample_results_df["k"], sample_results_df["davies_bouldin"], marker="o", color="tab:orange")
axes[1].set_title("Davies-Bouldin")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Índice DB")
axes[1].grid(alpha=0.3)

axes[2].plot(sample_results_df["k"], sample_results_df["sse"], marker="o", color="tab:green")
axes[2].set_title("Método del codo (SSE)")
axes[2].set_xlabel("k")
axes[2].set_ylabel("SSE")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nMejor k por silhouette:", int(sample_results_df.loc[sample_results_df["silhouette"].idxmax(), "k"]))
print("Mejor k por Davies-Bouldin:", int(sample_results_df.loc[sample_results_df["davies_bouldin"].idxmin(), "k"]))
print("El mejor k por SSE se identifica visualmente buscando el codo de la curva.")


### 1.1 Análisis y conclusión

- Con los resultados ya visibles del coeficiente de silueta, `k=2` obtiene el promedio más alto, pero eso no implica automáticamente que sea la mejor interpretación del problema.
- En estos datos sintéticos se generaron **4 centros**; por eso `k=4` suele ser la solución más coherente con la estructura real, mientras `k=2` puede estar fusionando grupos cercanos.
- La decisión final debe cruzar tres criterios: silhouette alto, Davies-Bouldin bajo y un codo claro en la curva de SSE.
- Si las tres métricas coinciden en `k=4`, esa sería la conclusión más sólida; si silhouette favorece `k=2`, explíquelo como un efecto de cercanía entre algunos grupos.

### 1.2 Davies-Bouldin

- El índice Davies-Bouldin se interpreta al revés de silhouette: **menor es mejor**.
- Úselo para verificar si la partición con grupos más compactos y mejor separados coincide con la sugerida por silhouette.

### 1.3 SSE y método del codo

- El SSE siempre disminuye cuando aumenta `k`, así que no se elige el mínimo sino el punto donde la mejora empieza a desacelerarse.
- Ese punto de inflexión es el codo y debe compararse con silhouette y Davies-Bouldin antes de reportar el valor final de `k`.


2. Cargar el conjunto de datos Iris y quitarle la etiqueta.

  2.1 Aplicar K-means con k=2,3,4,5 y 6.

  2.2 Validar con los indices SSE, Coeficiente de Silhouette y Davies_Bouldin

  2.3 Validar con medidas externas: Precisión, recall, F1, MI, y una de Entropia o Pureza.

In [ ]:
iris = load_iris(as_frame=True)
X_iris = iris.data.copy()
y_iris = iris.target.to_numpy()
target_names = iris.target_names
range_n_clusters = [2, 3, 4, 5, 6]

print("Dimensión de Iris sin etiqueta:", X_iris.shape)
print("Variables:", list(X_iris.columns))
display(X_iris.head())

def map_clusters_to_labels(y_true, cluster_labels):
    mapped_labels = np.empty_like(cluster_labels)
    cluster_to_label = {}

    for cluster in np.unique(cluster_labels):
        mask = cluster_labels == cluster
        majority_label = np.bincount(y_true[mask]).argmax()
        cluster_to_label[int(cluster)] = int(majority_label)
        mapped_labels[mask] = majority_label

    return mapped_labels, cluster_to_label

def purity_score(y_true, cluster_labels):
    total = 0
    for cluster in np.unique(cluster_labels):
        mask = cluster_labels == cluster
        total += np.bincount(y_true[mask]).max()
    return total / len(y_true)

iris_results = []

for k in range_n_clusters:
    model = KMeans(n_clusters=k, n_init=10, random_state=10)
    cluster_labels = model.fit_predict(X_iris)
    mapped_labels, cluster_mapping = map_clusters_to_labels(y_iris, cluster_labels)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_iris,
        mapped_labels,
        average="macro",
        zero_division=0,
    )

    iris_results.append(
        {
            "k": k,
            "sse": model.inertia_,
            "silhouette": silhouette_score(X_iris, cluster_labels),
            "davies_bouldin": davies_bouldin_score(X_iris, cluster_labels),
            "precision_macro": precision,
            "recall_macro": recall,
            "f1_macro": f1,
            "nmi": normalized_mutual_info_score(y_iris, cluster_labels),
            "purity": purity_score(y_iris, cluster_labels),
        }
    )

iris_results_df = pd.DataFrame(iris_results)
print("\nResumen de métricas para Iris:")
print(iris_results_df.round(4).to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
metrics_to_plot = [
    ("sse", "SSE (buscar codo)"),
    ("silhouette", "Silhouette"),
    ("davies_bouldin", "Davies-Bouldin"),
    ("precision_macro", "Precisión macro"),
    ("f1_macro", "F1 macro"),
    ("purity", "Pureza"),
]

for ax, (metric, title) in zip(axes.flat, metrics_to_plot):
    ax.plot(iris_results_df["k"], iris_results_df[metric], marker="o")
    ax.set_title(title)
    ax.set_xlabel("k")
    ax.set_ylabel(metric)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

best_k_silhouette = int(iris_results_df.loc[iris_results_df["silhouette"].idxmax(), "k"])
best_k_db = int(iris_results_df.loc[iris_results_df["davies_bouldin"].idxmin(), "k"])
best_k_f1 = int(iris_results_df.loc[iris_results_df["f1_macro"].idxmax(), "k"])

print("\nMejor k por silhouette:", best_k_silhouette)
print("Mejor k por Davies-Bouldin:", best_k_db)
print("Mejor k por F1 macro:", best_k_f1)

reference_k = 3
reference_model = KMeans(n_clusters=reference_k, n_init=10, random_state=10)
reference_clusters = reference_model.fit_predict(X_iris)
reference_labels, cluster_mapping = map_clusters_to_labels(y_iris, reference_clusters)

readable_mapping = {cluster: target_names[label] for cluster, label in cluster_mapping.items()}
print("\nCorrespondencia cluster -> especie para k=3:")
print(readable_mapping)

conf_matrix = confusion_matrix(y_iris, reference_labels)
confusion_df = pd.DataFrame(
    conf_matrix,
    index=[f"Real: {name}" for name in target_names],
    columns=[f"Pred: {name}" for name in target_names],
)
display(confusion_df)


### 2. Interpretación sugerida para Iris

- En Iris normalmente `k=3` es la solución natural porque el conjunto tiene tres especies reales.
- Aun así, puede pasar que silhouette o Davies-Bouldin favorezcan `k=2`; eso suele indicar que **versicolor** y **virginica** se solapan y K-means tiende a fusionarlas parcialmente.
- En las métricas externas, lo esperable es que **setosa** quede muy bien separada y que los errores se concentren entre versicolor y virginica.
- Para el informe, compare el mejor `k` según criterios internos con el mejor `k` según precisión, recall, F1, NMI y pureza, y explique cualquier diferencia entre estructura geométrica y clases reales.
- Si quiere una versión más robusta del experimento, puede repetirlo escalando las variables antes de aplicar K-means.
